In [1]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\rorym\AppData\Local\Programs\Python\Python314-arm64\python.exe -m pip install --upgrade pip


In [2]:
import duckdb

print(duckdb.__version__)

1.5.6


In [3]:
conn = duckdb.connect("procurement.duckdb")

In [4]:
conn.execute("""
    CREATE TABLE IF NOT EXISTS raw_releases (
        release_id VARCHAR PRIMARY KEY,
        ocid VARCHAR,
        release_date TIMESTAMP,
        tags JSON,
        raw_json JSON,
        ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

In [5]:
conn.execute("DESCRIBE raw_releases").fetchall()

[('release_id', 'VARCHAR', 'NO', 'PRI', None, None),
 ('ocid', 'VARCHAR', 'YES', None, None, None),
 ('release_date', 'TIMESTAMP WITH TIME ZONE', 'YES', None, None, None),
 ('tags', 'JSON', 'YES', None, None, None),
 ('raw_json', 'JSON', 'YES', None, None, None),
 ('ingested_at', 'TIMESTAMP', 'YES', None, 'CURRENT_TIMESTAMP', None)]

In [6]:
import requests
url = "https://www.find-tender.service.gov.uk/api/1.0/ocdsReleasePackages"

response = requests.get(url, timeout=10)
response.raise_for_status()

data = response.json()
releases = data["releases"]

print("Releases downloaded:", len(releases))

Releases downloaded: 100


In [7]:
print(releases[0]["ocid"])
print(releases[0]["id"])
print(releases[0]["date"])
print(releases[0]["tag"])

ocds-h6vhtk-078039
093578-2026
2026-10-04T11:13:59+01:00
['award', 'contract']


In [8]:
import json

tags_json = json.dumps(releases[0]["tag"])
raw_json = json.dumps(releases[0])

print(tags_json)
print(type(tags_json))

print(raw_json[:200])
print(type(raw_json))

["award", "contract"]
<class 'str'>
{"ocid": "ocds-h6vhtk-078039", "id": "093578-2026", "tag": ["award", "contract"], "date": "2026-10-04T11:13:59+01:00", "initiationType": "tender", "parties": [{"name": "British Council", "id": "GB-PPO
<class 'str'>


In [9]:
release = releases[0]

conn.execute("""
    INSERT INTO raw_releases (
        release_id,
        ocid,
        release_date,
        tags,
        raw_json
    )
    VALUES (?, ?, ?, ?, ?)
    ON CONFLICT (release_id) DO NOTHING
""", [
    release["id"],
    release["ocid"],
    release["date"],
    json.dumps(release["tag"]),
    json.dumps(release)
])

In [10]:
result = conn.execute("""
    SELECT
        release_id,
        ocid,
        release_date,
        tags,
        ingested_at
    FROM raw_releases
""").fetchall()

print(result)

[('092770-2026', 'ocds-h6vhtk-06759b', datetime.datetime(2026, 10, 1, 11, 46, 44, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>), '["award", "contract"]', datetime.datetime(2026, 10, 1, 12, 9, 56, 530408)), ('092769-2026', 'ocds-h6vhtk-077e7b', datetime.datetime(2026, 10, 1, 11, 46, 41, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>), '["tender"]', datetime.datetime(2026, 10, 1, 13, 54, 42, 273388)), ('092768-2026', 'ocds-h6vhtk-077e7a', datetime.datetime(2026, 10, 1, 11, 46, 10, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>), '["award", "contract"]', datetime.datetime(2026, 10, 1, 13, 54, 42, 296854)), ('092767-2026', 'ocds-h6vhtk-06ecf6', datetime.datetime(2026, 10, 1, 11, 45, 29, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>), '["planningUpdate"]', datetime.datetime(2026, 10, 1, 13, 54, 42, 310387)), ('092766-2026', 'ocds-h6vhtk-064e67', datetime.datetime(2026, 10, 1, 11, 44, 37, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>), '["contractTermination"]', datet

In [11]:
release = releases[0]

conn.execute("""
    INSERT INTO raw_releases (
        release_id,
        ocid,
        release_date,
        tags,
        raw_json
    )
    VALUES (?, ?, ?, ?, ?)
    ON CONFLICT (release_id) DO NOTHING
""", [
    release["id"],
    release["ocid"],
    release["date"],
    json.dumps(release["tag"]),
    json.dumps(release)
])

In [12]:
count = conn.execute("""
    SELECT COUNT(*)
    FROM raw_releases
""").fetchone()[0]

print("Rows:", count)

Rows: 1166


In [13]:
for release in releases:

    conn.execute("""
        INSERT INTO raw_releases (
            release_id,
            ocid,
            release_date,
            tags,
            raw_json
        )
        VALUES (?, ?, ?, ?, ?)
        ON CONFLICT (release_id) DO NOTHING
    """, [
        release["id"],
        release["ocid"],
        release["date"],
        json.dumps(release["tag"]),
        json.dumps(release)
    ])

In [14]:
count = conn.execute("""
    SELECT COUNT(*)
    FROM raw_releases
""").fetchone()[0]

print("Rows:", count)

Rows: 1166


In [15]:
url = "https://www.find-tender.service.gov.uk/api/1.0/ocdsReleasePackages"

page = 1
max_pages = 5

while url and page <= max_pages:

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    data = response.json()
    releases = data["releases"]

    for release in releases:

        conn.execute("""
        INSERT INTO raw_releases (
            release_id,
            ocid,
            release_date,
            tags,
            raw_json
        )
        VALUES (?, ?, ?, ?, ?)
        ON CONFLICT (release_id) DO NOTHING
    """, [
        release["id"],
        release["ocid"],
        release["date"],
        json.dumps(release["tag"]),
        json.dumps(release)
    ])

    url = data["links"].get("next")

    page += 1

In [16]:
count = conn.execute("""
    SELECT COUNT(*)
    FROM raw_releases
""").fetchone()[0]

print("Rows:", count)

Rows: 1166


In [17]:
url = "https://www.find-tender.service.gov.uk/api/1.0/ocdsReleasePackages"

page = 1
max_pages = 5

while url and page <= max_pages:

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    data = response.json()
    releases = data["releases"]

    for release in releases:

        conn.execute("""
        INSERT INTO raw_releases (
            release_id,
            ocid,
            release_date,
            tags,
            raw_json
        )
        VALUES (?, ?, ?, ?, ?)
        ON CONFLICT (release_id) DO NOTHING
    """, [
        release["id"],
        release["ocid"],
        release["date"],
        json.dumps(release["tag"]),
        json.dumps(release)
    ])

    url = data["links"].get("next")

    page += 1

In [18]:
count = conn.execute("""
    SELECT COUNT(*)
    FROM raw_releases
""").fetchone()[0]

print("Rows:", count)

Rows: 1166


In [19]:
conn.execute("""
    SELECT
        release_id,
        ocid,
        release_date,
        ingested_at
    FROM raw_releases
    ORDER BY ingested_at DESC
    LIMIT 10
""").fetchall()

[('093074-2026',
  'ocds-h6vhtk-077f22',
  datetime.datetime(2026, 10, 2, 7, 47, 47, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>),
  datetime.datetime(2026, 10, 4, 13, 25, 8, 254802)),
 ('093075-2026',
  'ocds-h6vhtk-06f5ce',
  datetime.datetime(2026, 10, 2, 7, 47, 55, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>),
  datetime.datetime(2026, 10, 4, 13, 25, 8, 252376)),
 ('093076-2026',
  'ocds-h6vhtk-067bb5',
  datetime.datetime(2026, 10, 2, 7, 57, 4, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>),
  datetime.datetime(2026, 10, 4, 13, 25, 8, 249567)),
 ('093077-2026',
  'ocds-h6vhtk-077f23',
  datetime.datetime(2026, 10, 2, 8, 2, 37, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>),
  datetime.datetime(2026, 10, 4, 13, 25, 8, 247267)),
 ('093078-2026',
  'ocds-h6vhtk-077f24',
  datetime.datetime(2026, 10, 2, 8, 5, 5, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>),
  datetime.datetime(2026, 10, 4, 13, 25, 8, 244881)),
 ('093079-2026',
  'ocds-h6vhtk-077f23',
  d

In [20]:
watermark = conn.execute("""
    SELECT MAX(release_date)
    FROM raw_releases
""").fetchone()[0]

print("Current watermark:", watermark)

Current watermark: 2026-10-04 11:13:59+01:00


In [21]:
watermark_str = watermark.isoformat()

print(watermark_str)

2026-10-04T11:13:59+01:00


In [22]:
url = "https://www.find-tender.service.gov.uk/api/1.0/ocdsReleasePackages"

params = {
    "updatedFrom": watermark_str
}

response = requests.get(
    url,
    params=params,
    timeout=10
)

response.raise_for_status()

incremental_data = response.json()
incremental_releases = incremental_data["releases"]

print("Request URL:", response.url)
print("Releases returned:", len(incremental_releases))

Request URL: https://www.find-tender.service.gov.uk/api/1.0/ocdsReleasePackages?updatedFrom=2026-10-04T11%3A13%3A59%2B01%3A00
Releases returned: 1


In [23]:
dates = [release["date"] for release in incremental_releases]

print("First:", dates[0])
print("Last:", dates[-1])
print("Minimum:", min(dates))
print("Maximum:", max(dates))
print("Next page:", incremental_data.get("links", {}).get("next"))

First: 2026-10-04T11:13:59+01:00
Last: 2026-10-04T11:13:59+01:00
Minimum: 2026-10-04T11:13:59+01:00
Maximum: 2026-10-04T11:13:59+01:00
Next page: None


In [24]:
conn.execute("""
    ALTER TABLE raw_releases
    ALTER COLUMN release_date TYPE TIMESTAMPTZ
""")

In [25]:
conn.execute("DESCRIBE raw_releases").fetchall()

[('release_id', 'VARCHAR', 'NO', 'PRI', None, None),
 ('ocid', 'VARCHAR', 'YES', None, None, None),
 ('release_date', 'TIMESTAMP WITH TIME ZONE', 'YES', None, None, None),
 ('tags', 'JSON', 'YES', None, None, None),
 ('raw_json', 'JSON', 'YES', None, None, None),
 ('ingested_at', 'TIMESTAMP', 'YES', None, 'CURRENT_TIMESTAMP', None)]

In [26]:
url = "https://www.find-tender.service.gov.uk/api/1.0/ocdsReleasePackages"

params = {
    "updatedFrom": watermark.isoformat()
}

incremental_releases = []
page = 1

while url:

    print(f"Downloading page {page}...")
    response = requests.get(
        url,
        params=params,
        timeout=10
    )
    response.raise_for_status()

    incremental_data = response.json()
    
    incremental_releases.extend(incremental_data["releases"])
    url = incremental_data.get("links", {}).get("next")
    params = None
    page += 1
    
print("Total incremental releases:", len(incremental_releases))

Total incremental releases: 1


In [27]:
print("Number of releases:", len(incremental_releases))

dates = [release["date"] for release in incremental_releases]

print("Earliest:", min(dates))
print("Latest:", max(dates))

Number of releases: 1
Earliest: 2026-10-04T11:13:59+01:00
Latest: 2026-10-04T11:13:59+01:00


In [28]:
existing_count = 0

for release in incremental_releases:
    result = conn.execute("""
        SELECT COUNT(*)
        FROM raw_releases
        WHERE release_id = ?
    """, [release["id"]]).fetchone()[0]

    if result > 0:
        existing_count += 1

print("Already in DuckDB:", existing_count)
print("Potentially new:", len(incremental_releases) - existing_count)

Already in DuckDB: 1
Potentially new: 0


In [29]:
url = "https://www.find-tender.service.gov.uk/api/1.0/ocdsReleasePackages"

params = {
    "updatedFrom": watermark.isoformat()
}

incremental_releases = []
page = 1
processed = 0
while url:

    print(f"Downloading page {page}...")
    response = requests.get(
        url,
        params=params,
        timeout=10
    )
    response.raise_for_status()

    incremental_data = response.json()

    for release in incremental_data["releases"]:
        processed += 1
        conn.execute("""
        INSERT INTO raw_releases (
            release_id,
            ocid,
            release_date,
            tags,
            raw_json
        )
        VALUES (?, ?, ?, ?, ?)
        ON CONFLICT (release_id) DO NOTHING
    """, [
        release["id"],
        release["ocid"],
        release["date"],
        json.dumps(release["tag"]),
        json.dumps(release)
    ])
    
    url = incremental_data.get("links", {}).get("next")
    params = None
    page += 1
    
print("processed:", processed)

processed: 1


In [30]:
count = conn.execute("""
    SELECT COUNT(*)
    FROM raw_releases
""").fetchone()[0]

print("Total rows:", count)

Total rows: 1166


In [63]:
## Refactoring the pipeline
def get_watermark():
    with duckdb.connect("procurement.duckdb") as conn:

        result = conn.execute("""
            SELECT watermark_to
            FROM pipeline_runs
            WHERE pipeline_name = 'procurement_ingestion'
              AND status = 'SUCCESS'
            ORDER BY completed_at DESC
            LIMIT 1
        """).fetchone()

        if result is not None:
            watermark = result[0]
        else:
            watermark = conn.execute("""
                SELECT MAX(release_date)
                FROM raw_releases
            """).fetchone()[0]

    return watermark

In [64]:
watermark = get_watermark()
print(watermark)

2026-10-04 14:29:57+01:00


In [36]:
def extract_releases(watermark):
    url = "https://www.find-tender.service.gov.uk/api/1.0/ocdsReleasePackages"

    params = {
        "updatedFrom": watermark.isoformat()
    }
    
    incremental_releases = []
    page = 1

    while url:
        response = requests.get(
            url,
            params=params,
            timeout=10
        )
        response.raise_for_status()
        print(f"Downloading page {page}...")
        incremental_data = response.json()
        incremental_releases.extend(incremental_data["releases"])
        
        
        url = incremental_data.get("links", {}).get("next")
        params = None
        page += 1

    return incremental_releases

In [37]:
watermark = get_watermark()

releases = extract_releases(watermark)

print("Extracted:", len(releases))

Extracted: 3


In [46]:
def load_releases(releases):
    with duckdb.connect("procurement.duckdb") as conn:
        before_count = conn.execute("""
        SELECT COUNT(*)
        FROM raw_releases
        """).fetchone()[0]
        processed = 0
        for release in releases:
            processed += 1
            conn.execute("""
            INSERT INTO raw_releases (
                release_id,
                ocid,
                release_date,
                tags,
                raw_json
            )
            VALUES (?, ?, ?, ?, ?)
            ON CONFLICT (release_id) DO NOTHING
            """, [
            release["id"],
            release["ocid"],
            release["date"],
            json.dumps(release["tag"]),
            json.dumps(release)
        ])
        after_count = conn.execute("""
        SELECT COUNT(*)
        FROM raw_releases
        """).fetchone()[0]
    inserted = after_count - before_count
    skipped = processed - inserted

    return processed, inserted, skipped

In [40]:
watermark = get_watermark()

releases = extract_releases(watermark)

processed = load_releases(releases)

print("Watermark:", watermark)
print("Extracted:", len(releases))
print("Processed:", processed)

Watermark: 2026-10-04 11:13:59+01:00
Extracted: 3
Processed: 3


In [41]:
count = duckdb.connect("procurement.duckdb").execute("""
    SELECT COUNT(*)
    FROM raw_releases
""").fetchone()[0]

print("Total rows:", count)

Total rows: 1168


In [47]:
watermark = get_watermark()
releases = extract_releases(watermark)
processed, inserted, skipped = load_releases(releases)

print("Processed:", processed)
print("Inserted:", inserted)
print("Skipped:", skipped)

Processed: 1
Inserted: 0
Skipped: 1


In [49]:
with duckdb.connect("procurement.duckdb") as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS pipeline_runs (
            run_id VARCHAR PRIMARY KEY,
            pipeline_name VARCHAR,
            started_at TIMESTAMPTZ,
            completed_at TIMESTAMPTZ,
            watermark_from TIMESTAMPTZ,
            watermark_to TIMESTAMPTZ,
            rows_extracted INTEGER,
            rows_inserted INTEGER,
            rows_skipped INTEGER,
            status VARCHAR
        )
    """)

In [50]:
with duckdb.connect("procurement.duckdb") as conn:
    result = conn.execute("""
        DESCRIBE pipeline_runs
    """).fetchall()

result

[('run_id', 'VARCHAR', 'NO', 'PRI', None, None),
 ('pipeline_name', 'VARCHAR', 'YES', None, None, None),
 ('started_at', 'TIMESTAMP WITH TIME ZONE', 'YES', None, None, None),
 ('completed_at', 'TIMESTAMP WITH TIME ZONE', 'YES', None, None, None),
 ('watermark_from', 'TIMESTAMP WITH TIME ZONE', 'YES', None, None, None),
 ('watermark_to', 'TIMESTAMP WITH TIME ZONE', 'YES', None, None, None),
 ('rows_extracted', 'INTEGER', 'YES', None, None, None),
 ('rows_inserted', 'INTEGER', 'YES', None, None, None),
 ('rows_skipped', 'INTEGER', 'YES', None, None, None),
 ('status', 'VARCHAR', 'YES', None, None, None)]

In [51]:
import uuid
run_id = str(uuid.uuid4())

print(run_id)

b49f4547-dd06-44e7-ba82-775a43ba9724


In [52]:
def start_pipeline_run(watermark):
    with duckdb.connect("procurement.duckdb") as conn:
        run_id = str(uuid.uuid4())
        conn.execute("""
            INSERT INTO pipeline_runs (
                run_id,
                pipeline_name,
                started_at,
                watermark_from,
                status
            )
            VALUES (?, ?, CURRENT_TIMESTAMP, ?, ?)
            """, [
            run_id,
            'procurement_ingestion',
            watermark,
            'RUNNING'
        ])
        return run_id
        

In [53]:
watermark = get_watermark()

run_id = start_pipeline_run(watermark)

print("Run ID:", run_id)

Run ID: 534c15cd-8f0c-4e1e-8d58-23400f8efbcc


In [54]:
with duckdb.connect("procurement.duckdb") as conn:
    result = conn.execute("""
        SELECT *
        FROM pipeline_runs
        WHERE run_id = ?
    """, [run_id]).fetchone()

print(result)

('534c15cd-8f0c-4e1e-8d58-23400f8efbcc', 'procurement_ingestion', datetime.datetime(2026, 10, 4, 19, 46, 19, 203101, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>), None, datetime.datetime(2026, 10, 4, 14, 29, 57, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>), None, None, None, None, 'RUNNING')


In [55]:
def complete_pipeline_run(
    run_id,
    watermark_to,
    rows_extracted,
    rows_inserted,
    rows_skipped
):
    with duckdb.connect("procurement.duckdb") as conn:
        conn.execute("""
            UPDATE pipeline_runs
            SET
                completed_at = CURRENT_TIMESTAMP,
                watermark_to = ?,
                rows_extracted = ?,
                rows_inserted = ?,
                rows_skipped = ?,
                status = 'SUCCESS'
            WHERE run_id = ?
            """, [
            watermark_to,
            rows_extracted,
            rows_inserted,
            rows_skipped,
            run_id
        ])

In [58]:
from datetime import datetime
if releases:
    watermark_to = max(
        datetime.fromisoformat(release["date"])
        for release in releases
    )
else:
    watermark_to = watermark

print(watermark_to)
print(type(watermark_to))

2026-10-04 14:29:57+01:00
<class 'datetime.datetime'>


In [59]:
with duckdb.connect("procurement.duckdb") as conn:
    running_run = conn.execute("""
        SELECT
            run_id,
            watermark_from
        FROM pipeline_runs
        WHERE status = 'RUNNING'
        ORDER BY started_at DESC
        LIMIT 1
    """).fetchone()

print(running_run)

('534c15cd-8f0c-4e1e-8d58-23400f8efbcc', datetime.datetime(2026, 10, 4, 14, 29, 57, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>))


In [60]:
run_id = running_run[0]
watermark = running_run[1]

print("Run ID:", run_id)
print("Watermark from:", watermark)

Run ID: 534c15cd-8f0c-4e1e-8d58-23400f8efbcc
Watermark from: 2026-10-04 14:29:57+01:00


In [61]:
complete_pipeline_run(
    run_id,
    watermark_to,
    len(releases),
    inserted,
    skipped
)

In [62]:
with duckdb.connect("procurement.duckdb") as conn:
    result = conn.execute("""
        SELECT *
        FROM pipeline_runs
        WHERE run_id = ?
    """, [run_id]).fetchone()

print(result)

('534c15cd-8f0c-4e1e-8d58-23400f8efbcc', 'procurement_ingestion', datetime.datetime(2026, 10, 4, 19, 46, 19, 203101, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>), datetime.datetime(2026, 10, 4, 20, 14, 59, 257281, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>), datetime.datetime(2026, 10, 4, 14, 29, 57, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>), datetime.datetime(2026, 10, 4, 14, 29, 57, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>), 1, 0, 1, 'SUCCESS')


In [83]:
def run_pipeline():

    watermark = get_watermark()
    print("Watermark from:", watermark)

    run_id = start_pipeline_run(watermark)
    print("Run ID:", run_id)

    try:
        releases = extract_releases(watermark)
        print("Extracted:", len(releases))

        processed, inserted, skipped = load_releases(releases)

        print("Processed:", processed)
        print("Inserted:", inserted)
        print("Skipped:", skipped)

        if releases:
            watermark_to = max(
                datetime.fromisoformat(release["date"])
                for release in releases
            )
        else:
            watermark_to = watermark

        print("Watermark to:", watermark_to)

        complete_pipeline_run(
            run_id,
            watermark_to,
            len(releases),
            inserted,
            skipped
        )
    
        print("Pipeline completed successfully")
    except Exception:
        fail_pipeline_run(run_id)
        print("Pipeline failed!")
        raise

In [84]:
run_pipeline()

Watermark from: 2026-10-04 18:52:05+01:00
Run ID: 24680ffe-814c-455d-925d-8ff5300ec7cc
Extracted: 1
Processed: 1
Inserted: 0
Skipped: 1
Watermark to: 2026-10-04 18:52:05+01:00
Pipeline completed successfully


In [85]:
with duckdb.connect("procurement.duckdb") as conn:
    results = conn.execute("""
        SELECT *
        FROM pipeline_runs
        ORDER BY started_at DESC
    """).df()

results

,run_id,pipeline_name,started_at,completed_at,watermark_from,watermark_to,rows_extracted,rows_inserted,rows_skipped,status
0,24680ffe-814c-455d-925d-8ff5300ec7cc,procurement_ingestion,2026-10-04 21:32:27.653057+01:00,2026-10-04 21:32:28.399111+01:00,2026-10-04 18:52:05+01:00,2026-10-04 18:52:05+01:00,1,0,1,SUCCESS
1,250a024e-4a9a-4022-8c56-3f0fba6bc702,procurement_ingestion,2026-10-04 21:19:27.707741+01:00,2026-10-04 21:19:27.712505+01:00,2026-10-04 18:52:05+01:00,NaT,<NA>,<NA>,<NA>,FAILED
2,30723c89-fd98-4fff-bdbc-5762b2700f9f,procurement_ingestion,2026-10-04 21:18:07.231051+01:00,NaT,2026-10-04 18:52:05+01:00,NaT,<NA>,<NA>,<NA>,RUNNING
3,2e250e06-17a0-45a0-90f8-aece2ab5df33,procurement_ingestion,2026-10-04 21:13:14.156645+01:00,2026-10-04 21:13:15.030846+01:00,2026-10-04 18:52:05+01:00,2026-10-04 18:52:05+01:00,1,0,1,SUCCESS
4,bd702418-d93d-4c0f-af25-75fe59c72076,procurement_ingestion,2026-10-04 21:12:49.326771+01:00,2026-10-04 21:12:50.137397+01:00,2026-10-04 18:52:05+01:00,2026-10-04 18:52:05+01:00,1,0,1,SUCCESS
5,334f7fc5-322b-4789-abf2-2e2ae3c404df,procurement_ingestion,2026-10-04 20:50:09.398674+01:00,2026-10-04 20:50:10.314728+01:00,2026-10-04 14:29:57+01:00,2026-10-04 18:52:05+01:00,3,2,1,SUCCESS
6,534c15cd-8f0c-4e1e-8d58-23400f8efbcc,procurement_ingestion,2026-10-04 19:46:19.203101+01:00,2026-10-04 20:14:59.257281+01:00,2026-10-04 14:29:57+01:00,2026-10-04 14:29:57+01:00,1,0,1,SUCCESS


In [82]:
def fail_pipeline_run(run_id):
    with duckdb.connect("procurement.duckdb") as conn:
        conn.execute("""
            UPDATE pipeline_runs
            SET
            completed_at = CURRENT_TIMESTAMP,
            status = 'FAILED'
            WHERE run_id = ?
            """, [
            run_id
            ])

In [86]:
print(get_watermark())

2026-10-04 18:52:05+01:00


In [81]:
with duckdb.connect("procurement.duckdb") as conn:
    recent_runs = conn.execute("""
        SELECT
            run_id,
            started_at,
            completed_at,
            watermark_from,
            watermark_to,
            rows_extracted,
            rows_inserted,
            rows_skipped,
            status
        FROM pipeline_runs
        ORDER BY started_at DESC
        LIMIT 5
    """).df()

recent_runs

,run_id,started_at,completed_at,watermark_from,watermark_to,rows_extracted,rows_inserted,rows_skipped,status
0,250a024e-4a9a-4022-8c56-3f0fba6bc702,2026-10-04 21:19:27.707741+01:00,2026-10-04 21:19:27.712505+01:00,2026-10-04 18:52:05+01:00,NaT,<NA>,<NA>,<NA>,FAILED
1,30723c89-fd98-4fff-bdbc-5762b2700f9f,2026-10-04 21:18:07.231051+01:00,NaT,2026-10-04 18:52:05+01:00,NaT,<NA>,<NA>,<NA>,RUNNING
2,2e250e06-17a0-45a0-90f8-aece2ab5df33,2026-10-04 21:13:14.156645+01:00,2026-10-04 21:13:15.030846+01:00,2026-10-04 18:52:05+01:00,2026-10-04 18:52:05+01:00,1,0,1,SUCCESS
3,bd702418-d93d-4c0f-af25-75fe59c72076,2026-10-04 21:12:49.326771+01:00,2026-10-04 21:12:50.137397+01:00,2026-10-04 18:52:05+01:00,2026-10-04 18:52:05+01:00,1,0,1,SUCCESS
4,334f7fc5-322b-4789-abf2-2e2ae3c404df,2026-10-04 20:50:09.398674+01:00,2026-10-04 20:50:10.314728+01:00,2026-10-04 14:29:57+01:00,2026-10-04 18:52:05+01:00,3,2,1,SUCCESS


In [2]:
conn.close()

NameError: name 'conn' is not defined

In [2]:
import requests
params = {
    "updatedFrom": "2026-10-04T18:52:05+01:00",
    "updatedTo": "2026-10-05T18:52:05+01:00"
}

response = requests.get(
    "https://www.find-tender.service.gov.uk/api/1.0/ocdsReleasePackages",
    params=params,
    timeout=30
)

print(response.status_code)

200


In [4]:
import duckdb
with duckdb.connect("procurement.duckdb") as conn:
    recent_runs = conn.execute("""
        SELECT
            run_id,
            started_at,
            completed_at,
            watermark_from,
            watermark_to,
            rows_extracted,
            rows_inserted,
            rows_skipped,
            status
        FROM pipeline_runs
        ORDER BY started_at DESC
        LIMIT 5
    """).df()

recent_runs

,run_id,started_at,completed_at,watermark_from,watermark_to,rows_extracted,rows_inserted,rows_skipped,status
0,4ef8c9df-a574-4a6a-8051-311edada9226,2026-10-22 21:05:50.844673+01:00,2026-10-22 21:05:51.694377+01:00,2026-10-22 20:56:12.372816+01:00,2026-10-22 21:05:50.899011+01:00,1,1,0,SUCCESS
1,4d72bf45-3442-4fb8-9bbc-78502cc80828,2026-10-22 20:56:12.318369+01:00,2026-10-22 21:04:06.339963+01:00,2026-10-04 18:52:05+01:00,2026-10-22 20:56:12.372816+01:00,6465,4200,2265,SUCCESS
2,1b126398-c802-4edf-8e33-0db8f070e39b,2026-10-22 20:48:33.729115+01:00,2026-10-22 20:51:48.001061+01:00,2026-10-04 18:52:05+01:00,NaT,<NA>,<NA>,<NA>,FAILED
3,1d43fdf9-a2df-4963-9c10-ea550a2df7a1,2026-10-22 20:38:41.330253+01:00,2026-10-22 20:39:38.339344+01:00,2026-10-04 18:52:05+01:00,NaT,<NA>,<NA>,<NA>,FAILED
4,9d09d4c5-d757-4566-8f95-61bc94962e51,2026-10-22 20:32:20.357029+01:00,2026-10-22 20:32:56.208632+01:00,2026-10-04 18:52:05+01:00,NaT,<NA>,<NA>,<NA>,FAILED
